
# 1. Le GPU et le choix du modèle

Le modèle est téléchargé au premier lancement dans HF_HOME. On le place dans ~/work, le dossier persistant du service, pour ne pas le retélécharger à chaque redémarrage. Cette ligne doit être exécutée avant tout import de transformers.


In [1]:

%pip install -q -U "transformers>=4.57" accelerate s3fs pillow pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["HF_HOME"] = os.path.expanduser("~/work/.cache/huggingface")

import torch

assert torch.cuda.is_available(), "Pas de GPU : relancer le service avec 1 GPU"
nom = torch.cuda.get_device_name(0)
memoire_go = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {nom} — {memoire_go:.0f} Go")

# Choix de la taille selon la memoire disponible (poids en bf16)
if memoire_go >= 75:
    MODELE = "Qwen/Qwen3-VL-32B-Instruct"     # ~66 Go
elif memoire_go >= 20:
    MODELE = "Qwen/Qwen3-VL-8B-Instruct"      # ~17 Go
else:
    MODELE = "Qwen/Qwen3-VL-4B-Instruct"      # ~9 Go
print("modèle retenu :", MODELE)
     


GPU : Tesla T4 — 16 Go
modèle retenu : Qwen/Qwen3-VL-4B-Instruct


# 2. Charger le modèle

Le premier chargement télécharge les poids : plusieurs minutes. Les suivants sont rapides.


In [3]:
import time
from transformers import AutoProcessor

try:
    from transformers import Qwen3VLForConditionalGeneration as Classe
except ImportError:
    from transformers import AutoModelForImageTextToText as Classe

t0 = time.time()
model = Classe.from_pretrained(MODELE, dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(MODELE)
model.eval()
print(f"chargé en {time.time() - t0:.0f} s")
import time
from transformers import AutoProcessor

try:
    from transformers import Qwen3VLForConditionalGeneration as Classe
except ImportError:
    from transformers import AutoModelForImageTextToText as Classe

t0 = time.time()
model = Classe.from_pretrained(MODELE, dtype="auto", device_map="auto")
processor = AutoProcessor.from_pretrained(MODELE)
model.eval()
print(f"chargé en {time.time() - t0:.0f} s")

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 713/713 [00:03<00:00, 230.56it/s]


chargé en 84 s


Loading weights: 100%|██████████| 713/713 [00:01<00:00, 358.58it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


chargé en 6 s


# 3. Interroger le modèle

do_sample=False : le modèle répond toujours la même chose à la même question. Indispensable pour mesurer sinon deux exécutions donnent deux taux différents.


In [4]:
import json, re

def demander(image, question, max_new_tokens=400):
    """Envoie une image et une question au modele, renvoie le texte produit."""
    chemin_tmp = "/tmp/page_courante.png"
    image.save(chemin_tmp)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": chemin_tmp},
        {"type": "text", "text": question},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        sortie = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    sortie = sortie[:, inputs["input_ids"].shape[1]:]      # on retire la question
    return processor.batch_decode(sortie, skip_special_tokens=True)[0].strip()

def extraire_json(texte):
    """Recupere le premier objet ou la premiere liste JSON dans la reponse."""
    m = re.search(r"(\{.*\}|\[.*\])", texte, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None

# 4. Les données

Mêmes images que le notebook 01. On garde la métropole : c'est le seul périmètre comparable au Fiqual de type 1.


In [5]:
import io
from pathlib import Path

import pandas as pd
import s3fs
from PIL import Image

BUCKET = "projet-mesure-qualite-rp"
CAMPAGNE = "RG25"
fs = s3fs.S3FileSystem()

fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_LOG.jpg")]
logs = pd.DataFrame({"chemin": fichiers})
logs["nom"] = logs.chemin.map(lambda c: Path(c).stem)
logs["dep"] = logs.chemin.str.split("/").str[6]
morceaux = logs.nom.str.split("_")
logs["cle"] = morceaux.str[0] + "_" + morceaux.str[1]

DOM = {"971", "972", "973", "974", "975", "976", "977", "978"}
logs = logs[~logs.dep.isin(DOM)].reset_index(drop=True)
print(len(logs), "pages LOG de métropole")


def charger(chemin):
    with fs.open(chemin, "rb") as f:
        return Image.open(io.BytesIO(f.read())).convert("RGB")

129 pages LOG de métropole


# 5. La règle des cases

Identique au notebook 01. En mode A, c'est elle qui transforme les états vus par le modèle en réponse finale.


In [6]:
FORCE = ("cochee", "biffee", "coloree")
ETATS = {"vide", "cochee", "biffee", "coloree", "autre"}
MODALITE_BASSE = {"BI_Q10", "BI_Q11"}
BLANC_SI_PLUSIEURS = {"BI_Q9", "FL_Q3", "FL_Q7", "FL_Q13",
                      "FLDOM_Q3", "FLDOM_Q7", "FLDOM_Q11", "FLDOM_Q14",
                      "TPSRESALT", "AUTRECOMETU"}


def valeur_case(etat, gagnant):
    if etat == "vide":
        return "0"
    if gagnant is not None and etat == gagnant:
        return "1"
    if gagnant == "cochee" and etat == "biffee":
        return "2"
    return "3"


def regle_cases(question, etats):
    gagnant = next((f for f in FORCE if f in etats), None)
    detail = "".join(valeur_case(e, gagnant) for e in etats)
    positions = [i + 1 for i, c in enumerate(detail) if c == "1"]
    if not positions:
        return "", detail
    if question in BLANC_SI_PLUSIEURS and len(positions) > 1:
        return "", detail
    if question in MODALITE_BASSE:
        return str(min(positions)), detail
    return str(max(positions)), detail

# 6. Les questions posées au modèle

Les définitions des cinq états reprennent la section 1.2 du document de consignes. À relire avec les consignes : la formulation exacte compte beaucoup pour un modèle de langage.


In [2]:
PROMPT_OEIL = """Voici la page 4 d'une feuille de logement du recensement.
Regarde uniquement la question 1 « Type de logement ». Elle a six cases, dans cet ordre :
1 Maison, 2 Appartement, 3 Logement-foyer, 4 Chambre d'hôtel,
5 Habitation de fortune, 6 Pièce indépendante.

Pour chaque case, donne son état parmi :
- vide : aucune marque dans la case. Un grand trait qui traverse une partie du questionnaire compte comme vide.
- cochee : une croix dans la case, ou une case atteinte par une croix.
- biffee : case à demi cochée, par exemple un seul trait.
- coloree : case entièrement remplie.
- autre : case raturée, gribouillée, ou tout autre marquage.

Réponds uniquement avec une liste JSON de six états, dans l'ordre des cases.
Exemple : ["vide", "cochee", "vide", "vide", "vide", "vide"]"""


PROMPT_TOUT = """Voici la page 4 d'une feuille de logement du recensement.
Donne la réponse à la question 1 « Type de logement » : 1 Maison, 2 Appartement,
3 Logement-foyer, 4 Chambre d'hôtel, 5 Habitation de fortune, 6 Pièce indépendante.

Règles de saisie :
- Si une seule case est cochée, réponds son numéro.
- Si plusieurs cases sont marquées, le marquage le plus fort l'emporte :
  une croix, puis un demi-cochage, puis une case coloriée. Une case raturée ne compte pas.
- Si plusieurs cases ont ce marquage le plus fort, réponds le numéro le plus élevé.
- Si aucune case n'est marquée, réponds une chaîne vide.

Réponds uniquement en JSON : {"TYPL": "2"}"""

# 7. Test sur une seule image
On commence par les limites du notebook 01

In [3]:
CHEMIN_TEST = None     

chemin = CHEMIN_TEST or logs.chemin.iloc[0]
page = charger(chemin)
print(chemin)

t0 = time.time()
brut_a = demander(page, PROMPT_OEIL)
etats = extraire_json(brut_a)
print(f"\nMODE A ({time.time() - t0:.1f} s)")
print("réponse brute :", brut_a)
if isinstance(etats, list) and len(etats) == 6 and set(etats) <= ETATS:
    print("CHOIX, DETAIL :", regle_cases("FL_Q1", etats))
else:
    print("réponse inutilisable")

t0 = time.time()
brut_b = demander(page, PROMPT_TOUT)
print(f"\nMODE B ({time.time() - t0:.1f} s)")
print("réponse brute :", brut_b)
print("TYPL :", (extraire_json(brut_b) or {}).get("TYPL"))

NameError: name 'logs' is not defined